# [9.2] Chain-of-Thought Faithfulness - Exercises

Implement the local report functions for the CoT faithfulness harness, then run the visible tests.


In [ ]:
from dataclasses import dataclass
import json
import sys
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter9_alignment_interpretability"
section = "part2_cot_faithfulness"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_cot_faithfulness.tests as tests
import part2_cot_faithfulness.utils as utils

MAIN = __name__ == "__main__"
CoTCondition = Literal["no_cot", "faithful_cot", "biased_cot", "posthoc"]

GT_TIER = "GT-3"
EXERCISE_ID = "9_2_chain_of_thought_faithfulness"
EXPECTED_RUNTIME = "45-60 minutes for exercises; a few minutes for the pinned CUDA preflight"
REQUIRES_GPU = True


In [ ]:
@dataclass(frozen=True)
class PreFinalAnswerProbeReport:
    hidden_answer_accuracy: float
    final_answer_agreement: float
    predicts_hidden_answer: bool


@dataclass(frozen=True)
class HiddenAnswerPatchingReport:
    original_answer: int
    patched_answer: int
    changed_output: bool


@dataclass(frozen=True)
class CoTTextBaselineReport:
    detector_recall: float
    text_only_recall: float
    text_only_misses_cases: bool


@dataclass(frozen=True)
class FeatureDetectorReport:
    feature_accuracy: float
    baseline_accuracy: float
    improves_detection: bool


@dataclass(frozen=True)
class CoTConditionComparisonReport:
    condition_accuracies: dict[str, float]
    biased_gap: float
    posthoc_gap: float


In [ ]:
def prediction_accuracy(logits: t.Tensor, target_token_ids: t.Tensor) -> float:
    raise NotImplementedError()


tests.test_prediction_accuracy_checks_top1_predictions(prediction_accuracy)


In [ ]:
def pre_final_answer_probe_report(
    probe_logits: t.Tensor,
    hidden_answer_ids: t.Tensor,
    final_answer_ids: t.Tensor,
    *,
    min_hidden_accuracy: float = 0.8,
) -> PreFinalAnswerProbeReport:
    raise NotImplementedError()


tests.test_pre_final_answer_probe_report_predicts_hidden_answer(
    pre_final_answer_probe_report,
)


In [ ]:
def hidden_answer_patching_report(
    original_answer_logits: t.Tensor,
    patched_answer_logits: t.Tensor,
) -> HiddenAnswerPatchingReport:
    raise NotImplementedError()


tests.test_hidden_answer_patching_report_flags_answer_flip(
    hidden_answer_patching_report,
)


In [ ]:
def _binary_recall(predictions: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def cot_text_baseline_report(
    detector_predictions: t.Tensor,
    text_only_predictions: t.Tensor,
    unfaithful_labels: t.Tensor,
) -> CoTTextBaselineReport:
    raise NotImplementedError()


tests.test_cot_text_baseline_report_keeps_recall_gap(cot_text_baseline_report)


In [ ]:
def feature_detector_report(
    feature_scores: t.Tensor,
    baseline_scores: t.Tensor,
    unfaithful_labels: t.Tensor,
    *,
    threshold: float = 0.5,
) -> FeatureDetectorReport:
    raise NotImplementedError()


tests.test_feature_detector_report_scores_thresholded_predictions(
    feature_detector_report,
)


In [ ]:
def cot_condition_comparison_report(
    condition_correct: dict[CoTCondition, t.Tensor],
) -> CoTConditionComparisonReport:
    raise NotImplementedError()


tests.test_cot_condition_comparison_report_tracks_gaps(
    cot_condition_comparison_report,
)


In [ ]:
def probe_smoke_test() -> dict:
    probe_logits = t.tensor([[2.0, 0.0], [0.0, 2.0], [2.0, 0.0]])
    hidden_answer_ids = t.tensor([0, 1, 0])
    final_answer_ids = t.tensor([0, 0, 0])
    return pre_final_answer_probe_report(
        probe_logits,
        hidden_answer_ids,
        final_answer_ids,
        min_hidden_accuracy=1.0,
    ).__dict__


def patching_smoke_test() -> dict:
    original_logits = t.tensor([3.0, 0.0])
    patched_logits = t.tensor([0.0, 3.0])
    return hidden_answer_patching_report(original_logits, patched_logits).__dict__


def text_baseline_smoke_test() -> dict:
    labels = t.tensor([1, 0, 1, 0], dtype=t.bool)
    detector = t.tensor([1, 0, 1, 0], dtype=t.bool)
    text_only = t.tensor([0, 0, 1, 0], dtype=t.bool)
    return cot_text_baseline_report(detector, text_only, labels).__dict__


def feature_detector_smoke_test() -> dict:
    labels = t.tensor([1, 0, 1, 0], dtype=t.bool)
    feature_scores = t.tensor([0.9, 0.1, 0.8, 0.2])
    baseline_scores = t.tensor([0.2, 0.1, 0.6, 0.2])
    return feature_detector_report(
        feature_scores,
        baseline_scores,
        labels,
        threshold=0.5,
    ).__dict__


def condition_comparison_smoke_test() -> dict:
    return cot_condition_comparison_report(
        {
            "no_cot": t.tensor([1, 0, 1], dtype=t.float32),
            "faithful_cot": t.tensor([1, 1, 1], dtype=t.float32),
            "biased_cot": t.tensor([1, 0, 0], dtype=t.float32),
            "posthoc": t.tensor([1, 1, 0], dtype=t.float32),
        }
    ).__dict__


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "probe": probe_smoke_test(),
        "patching": patching_smoke_test(),
        "text_baseline": text_baseline_smoke_test(),
        "feature_detector": feature_detector_smoke_test(),
        "condition_comparison": condition_comparison_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    tests.test_committed_gpu_report_uses_real_text_only_baseline(gpu)
    assert report["accepted"] and report["tests_passed"]
    assert report["gt_tier"] == "GT-3"
    assert gpu["preflight_passed"]
    assert gpu["model_name"] == "EleutherAI/pythia-70m-deduped"
    assert gpu["hidden_answer_accuracy"] == 1.0
    assert gpu["label_shuffled_probe_accuracy"] == 0.0
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
{key: gpu[key] for key in [
    "device",
    "hidden_answer_accuracy",
    "final_answer_agreement",
    "label_shuffled_probe_accuracy",
    "peak_vram_gb",
]}
